# Import

In [3]:
# Import der wichtigen Pakete
import os
import pandas as pd
from bs4 import BeautifulSoup
import sqlite3
import tarfile
from datetime import datetime
import requests
import gzip
import re
import matplotlib.pyplot as plt
from collections import defaultdict
from charset_normalizer import detect
import matplotlib.dates as mdates
import json
import zipfile
import tarfile
import numpy as np
import ast


from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier


In [8]:
df = pd.read_csv(r'C:\Users\maxtr\OneDrive\Studium\Data_Mining_Arbeit\Data-Mining-Project\200125_LoL_champion_data.csv')

# EDA

In [ ]:
df.shape

(172, 33)

In [5]:
df.size

5676

In [ ]:
df.head(8)

,Unnamed: 0,id,apiname,title,difficulty,herotype,alttype,resource,stats,rangetype,...,be,rp,skill_i,skill_q,skill_w,skill_e,skill_r,skills,fullname,nickname
0,Aatrox,266.0,Aatrox,the Darkin Blade,2,Fighter,Tank,Blood Well,"{'hp_base': 650, 'hp_lvl': 114, 'mp_base': 0, ...",Melee,...,4800,880,{1: 'Deathbringer Stance'},"{1: 'The Darkin Blade', 2: 'The Darkin Blade 3'}",{1: 'Infernal Chains'},{1: 'Umbral Dash'},{1: 'World Ender'},"{1: 'Deathbringer Stance', 2: 'The Darkin Blad...",NaN,NaN
1,Ahri,103.0,Ahri,the Nine-Tailed Fox,2,Mage,Assassin,Mana,"{'hp_base': 590, 'hp_lvl': 104, 'mp_base': 418...",Ranged,...,3150,790,{1: 'Essence Theft'},{1: 'Orb of Deception'},{1: 'Fox-Fire'},{1: 'Charm'},{1: 'Spirit Rush'},"{1: 'Essence Theft', 2: 'Orb of Deception', 3:...",NaN,NaN
2,Akali,84.0,Akali,the Rogue Assassin,2,Assassin,NaN,Energy,"{'hp_base': 600, 'hp_lvl': 119, 'mp_base': 200...",Melee,...,3150,790,"{1: ""Assassin's Mark""}",{1: 'Five Point Strike'},{1: 'Twilight Shroud'},{1: 'Shuriken Flip'},{1: 'Perfect Execution'},"{1: ""Assassin's Mark"", 2: 'Five Point Strike',...",Akali Jhomen Tethi,NaN
3,Akshan,166.0,Akshan,the Rogue Sentinel,3,Marksman,Assassin,Mana,"{'hp_base': 630, 'hp_lvl': 107, 'mp_base': 350...",Ranged,...,4800,880,{1: 'Dirty Fighting'},{1: 'Avengerang'},{1: 'Going Rogue'},{1: 'Heroic Swing'},{1: 'Comeuppance'},"{1: 'Dirty Fighting', 2: 'Avengerang', 3: 'Goi...",NaN,NaN
4,Alistar,12.0,Alistar,the Minotaur,1,Tank,Support,Mana,"{'hp_base': 685, 'hp_lvl': 120, 'mp_base': 350...",Melee,...,1350,585,{1: 'Triumphant Roar'},{1: 'Pulverize'},{1: 'Headbutt'},{1: 'Trample'},{1: 'Unbreakable Will'},"{1: 'Triumphant Roar', 2: 'Pulverize', 3: 'Hea...",NaN,NaN
5,Ambessa,799.0,Ambessa,The Matriarch of War,3,Fighter,Assassin,Energy,"{'hp_base': 630, 'hp_lvl': 110, 'mp_base': 200...",Melee,...,7800,975,"{1: ""Drakehound's Step""}","{1: 'Cunning Sweep', 2: 'Sundering Slam'}",{1: 'Repudiation'},{1: 'Lacerate'},{1: 'Public Execution'},"{1: ""Drakehound's Step"", 2: 'Cunning Sweep', 3...",Ambessa Medarda,NaN
6,Amumu,32.0,Amumu,the Sad Mummy,1,Tank,Support,Mana,"{'hp_base': 685, 'hp_lvl': 94, 'mp_base': 285,...",Melee,...,450,260,{1: 'Cursed Touch'},{1: 'Bandage Toss'},{1: 'Despair'},{1: 'Tantrum'},{1: 'Curse of the Sad Mummy'},"{1: 'Cursed Touch', 2: 'Bandage Toss', 3: 'Des...",NaN,NaN


In [7]:
df.columns

Index(['Unnamed: 0', 'id', 'apiname', 'title', 'difficulty', 'herotype',
       'alttype', 'resource', 'stats', 'rangetype', 'date', 'patch', 'changes',
       'role', 'client_positions', 'external_positions', 'damage', 'toughness',
       'control', 'mobility', 'utility', 'style', 'adaptivetype', 'be', 'rp',
       'skill_i', 'skill_q', 'skill_w', 'skill_e', 'skill_r', 'skills',
       'fullname', 'nickname'],
      dtype='object')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 33 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          172 non-null    object 
 1   id                  172 non-null    float64
 2   apiname             172 non-null    object 
 3   title               172 non-null    object 
 4   difficulty          172 non-null    int64  
 5   herotype            172 non-null    object 
 6   alttype             144 non-null    object 
 7   resource            167 non-null    object 
 8   stats               172 non-null    object 
 9   rangetype           172 non-null    object 
 10  date                172 non-null    object 
 11  patch               172 non-null    object 
 12  changes             172 non-null    object 
 13  role                172 non-null    object 
 14  client_positions    172 non-null    object 
 15  external_positions  172 non-null    object 
 16  damage  

In [9]:
df.describe()

,id,difficulty,damage,toughness,control,mobility,utility,style,be,rp
count,172.000000,172.000000,172.000000,172.000000,172.000000,172.000000,172.000000,172.000000,172.000000,172.000000
mean,195.728488,1.866279,2.465116,1.616279,1.994186,1.790698,1.453488,59.686047,3229.854651,711.773256
std,243.326685,0.666335,0.695813,0.759634,0.721411,0.781605,0.669415,32.094437,1784.829203,221.286564
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,450.000000,260.000000
25%,43.750000,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,30.000000,1350.000000,585.000000
50%,102.500000,2.000000,3.000000,1.000000,2.000000,2.000000,1.000000,65.000000,3150.000000,790.000000
75%,233.250000,2.000000,3.000000,2.000000,3.000000,2.000000,2.000000,90.000000,4800.000000,880.000000
max,950.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,100.000000,7800.000000,975.000000


In [10]:
df['herotype'].value_counts()

herotype
Fighter     49
Mage        37
Marksman    28
Tank        24
Assassin    17
Support     17
Name: count, dtype: int64

In [11]:
df['difficulty'].value_counts()

difficulty
2    93
1    51
3    28
Name: count, dtype: int64

In [12]:
df['role'].value_counts()

role
{'Marksman'}                  21
{'Vanguard'}                  16
{'Diver'}                     15
{'Juggernaut'}                14
{'Specialist'}                14
{'Skirmisher'}                14
{'Assassin'}                  13
{'Burst'}                     12
{'Battlemage'}                11
{'Enchanter'}                  8
{'Catcher'}                    7
{'Artillery'}                  5
{'Warden'}                     5
{'Enchanter', 'Burst'}         2
{'Assassin', 'Diver'}          2
{'Burst', 'Assassin'}          1
{'Skirmisher', 'Diver'}        1
{'Marksman', 'Catcher'}        1
{'Warden', 'Skirmisher'}       1
{'Assassin', 'Marksman'}       1
{'Burst', 'Artillery'}         1
{'Burst', 'Catcher'}           1
{'Assassin', 'Catcher'}        1
{'Enchanter', 'Marksman'}      1
{'Burst', 'Skirmisher'}        1
{'Enchanter', 'Warden'}        1
{'Marksman', 'Artillery'}      1
{'Assassin', 'Skirmisher'}     1
Name: count, dtype: int64

In [13]:
df['rangetype'].value_counts()

rangetype
Melee     91
Ranged    81
Name: count, dtype: int64

# Data Preparation

In [14]:
df.isnull().sum()  # Anzahl der fehlenden Werte pro Spalte


Unnamed: 0              0
id                      0
apiname                 0
title                   0
difficulty              0
herotype                0
alttype                28
resource                5
stats                   0
rangetype               0
date                    0
patch                   0
changes                 0
role                    0
client_positions        0
external_positions      0
damage                  0
toughness               0
control                 0
mobility                0
utility                 0
style                   0
adaptivetype            0
be                      0
rp                      0
skill_i                 0
skill_q                 0
skill_w                 0
skill_e                 0
skill_r                 0
skills                  2
fullname              129
nickname              158
dtype: int64

In [15]:
df.isna().mean() * 100  # Prozentuale Anzahl der fehlenden Werte


Unnamed: 0             0.000000
id                     0.000000
apiname                0.000000
title                  0.000000
difficulty             0.000000
herotype               0.000000
alttype               16.279070
resource               2.906977
stats                  0.000000
rangetype              0.000000
date                   0.000000
patch                  0.000000
changes                0.000000
role                   0.000000
client_positions       0.000000
external_positions     0.000000
damage                 0.000000
toughness              0.000000
control                0.000000
mobility               0.000000
utility                0.000000
style                  0.000000
adaptivetype           0.000000
be                     0.000000
rp                     0.000000
skill_i                0.000000
skill_q                0.000000
skill_w                0.000000
skill_e                0.000000
skill_r                0.000000
skills                 1.162791
fullname

In [16]:
df[df.isnull().any(axis=1)]  # Zeilen mit mindestens einem NaN-Wert anzeigen


,Unnamed: 0,id,apiname,title,difficulty,herotype,alttype,resource,stats,rangetype,...,be,rp,skill_i,skill_q,skill_w,skill_e,skill_r,skills,fullname,nickname
0,Aatrox,266.0,Aatrox,the Darkin Blade,2,Fighter,Tank,Blood Well,"{'hp_base': 650, 'hp_lvl': 114, 'mp_base': 0, ...",Melee,...,4800,880,{1: 'Deathbringer Stance'},"{1: 'The Darkin Blade', 2: 'The Darkin Blade 3'}",{1: 'Infernal Chains'},{1: 'Umbral Dash'},{1: 'World Ender'},"{1: 'Deathbringer Stance', 2: 'The Darkin Blad...",NaN,NaN
1,Ahri,103.0,Ahri,the Nine-Tailed Fox,2,Mage,Assassin,Mana,"{'hp_base': 590, 'hp_lvl': 104, 'mp_base': 418...",Ranged,...,3150,790,{1: 'Essence Theft'},{1: 'Orb of Deception'},{1: 'Fox-Fire'},{1: 'Charm'},{1: 'Spirit Rush'},"{1: 'Essence Theft', 2: 'Orb of Deception', 3:...",NaN,NaN
2,Akali,84.0,Akali,the Rogue Assassin,2,Assassin,NaN,Energy,"{'hp_base': 600, 'hp_lvl': 119, 'mp_base': 200...",Melee,...,3150,790,"{1: ""Assassin's Mark""}",{1: 'Five Point Strike'},{1: 'Twilight Shroud'},{1: 'Shuriken Flip'},{1: 'Perfect Execution'},"{1: ""Assassin's Mark"", 2: 'Five Point Strike',...",Akali Jhomen Tethi,NaN
3,Akshan,166.0,Akshan,the Rogue Sentinel,3,Marksman,Assassin,Mana,"{'hp_base': 630, 'hp_lvl': 107, 'mp_base': 350...",Ranged,...,4800,880,{1: 'Dirty Fighting'},{1: 'Avengerang'},{1: 'Going Rogue'},{1: 'Heroic Swing'},{1: 'Comeuppance'},"{1: 'Dirty Fighting', 2: 'Avengerang', 3: 'Goi...",NaN,NaN
4,Alistar,12.0,Alistar,the Minotaur,1,Tank,Support,Mana,"{'hp_base': 685, 'hp_lvl': 120, 'mp_base': 350...",Melee,...,1350,585,{1: 'Triumphant Roar'},{1: 'Pulverize'},{1: 'Headbutt'},{1: 'Trample'},{1: 'Unbreakable Will'},"{1: 'Triumphant Roar', 2: 'Pulverize', 3: 'Hea...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,Zeri,221.0,Zeri,The Spark of Zaun,2,Marksman,NaN,Mana,"{'hp_base': 600, 'hp_lvl': 110, 'mp_base': 250...",Ranged,...,4800,880,{1: 'Living Battery'},{1: 'Burst Fire'},{1: 'Ultrashock Laser'},{1: 'Spark Surge'},{1: 'Lightning Crash'},"{1: 'Living Battery', 2: 'Burst Fire', 3: 'Ult...",NaN,NaN
168,Ziggs,115.0,Ziggs,the Hexplosives Expert,2,Mage,NaN,Mana,"{'hp_base': 606, 'hp_lvl': 106, 'mp_base': 480...",Ranged,...,4800,880,{1: 'Short Fuse'},{1: 'Bouncing Bomb'},{1: 'Satchel Charge'},{1: 'Hexplosive Minefield'},{1: 'Mega Inferno Bomb'},"{1: 'Short Fuse', 2: 'Bouncing Bomb', 3: 'Satc...",NaN,NaN
169,Zilean,26.0,Zilean,the Chronokeeper,2,Support,Mage,Mana,"{'hp_base': 574, 'hp_lvl': 96, 'mp_base': 452,...",Ranged,...,1350,585,{1: 'Time in a Bottle'},{1: 'Time Bomb'},{1: 'Rewind'},{1: 'Time Warp'},{1: 'Chronoshift'},"{1: 'Time in a Bottle', 2: 'Time Bomb', 3: 'Re...",NaN,NaN
170,Zoe,142.0,Zoe,the Aspect of Twilight,3,Mage,Support,Mana,"{'hp_base': 630, 'hp_lvl': 106, 'mp_base': 425...",Ranged,...,4800,880,{1: 'More Sparkles!'},"{1: 'Paddle Star', 2: 'Paddle Star 2'}",{1: 'Spell Thief'},{1: 'Sleepy Trouble Bubble'},{1: 'Portal Jump'},"{1: 'More Sparkles!', 2: 'Paddle Star', 3: 'Sp...",NaN,NaN


In [17]:
columns_to_drop = ['fullname', 'nickname', 'skill_i','skill_q','skill_w','skill_e','skill_r','skills', 'Unnamed: 0', 'id', 'apiname', 'title', 'date', 'patch', 'changes']
df = df.drop(columns=columns_to_drop, errors='ignore')

In [18]:
df.head()

,difficulty,herotype,alttype,resource,stats,rangetype,role,client_positions,external_positions,damage,toughness,control,mobility,utility,style,adaptivetype,be,rp
0,2,Fighter,Tank,Blood Well,"{'hp_base': 650, 'hp_lvl': 114, 'mp_base': 0, ...",Melee,{'Juggernaut'},{'Top'},{'Top'},3,3,2,2,2,20,Physical,4800,880
1,2,Mage,Assassin,Mana,"{'hp_base': 590, 'hp_lvl': 104, 'mp_base': 418...",Ranged,{'Burst'},{'Middle'},{'Middle'},3,1,2,3,1,100,Magic,3150,790
2,2,Assassin,NaN,Energy,"{'hp_base': 600, 'hp_lvl': 119, 'mp_base': 200...",Melee,{'Assassin'},"{'Middle', 'Top'}","{'Middle', 'Top'}",3,1,1,3,1,65,Physical,3150,790
3,3,Marksman,Assassin,Mana,"{'hp_base': 630, 'hp_lvl': 107, 'mp_base': 350...",Ranged,"{'Assassin', 'Marksman'}",{'Middle'},{'Middle'},3,1,1,3,2,1,Physical,4800,880
4,1,Tank,Support,Mana,"{'hp_base': 685, 'hp_lvl': 120, 'mp_base': 350...",Melee,{'Vanguard'},{'Support'},{'Support'},1,3,3,1,2,65,Magic,1350,585


In [19]:
df.isnull().sum() 


difficulty             0
herotype               0
alttype               28
resource               5
stats                  0
rangetype              0
role                   0
client_positions       0
external_positions     0
damage                 0
toughness              0
control                0
mobility               0
utility                0
style                  0
adaptivetype           0
be                     0
rp                     0
dtype: int64

In [20]:
df['resource'].fillna('no resource', inplace=True)
df['alttype'].fillna('no 2nd type', inplace=True)

df.isnull().sum() 


difficulty            0
herotype              0
alttype               0
resource              0
stats                 0
rangetype             0
role                  0
client_positions      0
external_positions    0
damage                0
toughness             0
control               0
mobility              0
utility               0
style                 0
adaptivetype          0
be                    0
rp                    0
dtype: int64

In [21]:
df.duplicated().sum()

0

In [22]:
df.head()

,difficulty,herotype,alttype,resource,stats,rangetype,role,client_positions,external_positions,damage,toughness,control,mobility,utility,style,adaptivetype,be,rp
0,2,Fighter,Tank,Blood Well,"{'hp_base': 650, 'hp_lvl': 114, 'mp_base': 0, ...",Melee,{'Juggernaut'},{'Top'},{'Top'},3,3,2,2,2,20,Physical,4800,880
1,2,Mage,Assassin,Mana,"{'hp_base': 590, 'hp_lvl': 104, 'mp_base': 418...",Ranged,{'Burst'},{'Middle'},{'Middle'},3,1,2,3,1,100,Magic,3150,790
2,2,Assassin,no 2nd type,Energy,"{'hp_base': 600, 'hp_lvl': 119, 'mp_base': 200...",Melee,{'Assassin'},"{'Middle', 'Top'}","{'Middle', 'Top'}",3,1,1,3,1,65,Physical,3150,790
3,3,Marksman,Assassin,Mana,"{'hp_base': 630, 'hp_lvl': 107, 'mp_base': 350...",Ranged,"{'Assassin', 'Marksman'}",{'Middle'},{'Middle'},3,1,1,3,2,1,Physical,4800,880
4,1,Tank,Support,Mana,"{'hp_base': 685, 'hp_lvl': 120, 'mp_base': 350...",Melee,{'Vanguard'},{'Support'},{'Support'},1,3,3,1,2,65,Magic,1350,585


# Modeling

In [ ]:
# Data Mining, um herauszufinden, welche Attribute der nächste Champion haben könnte.


# Evaluation

# Review